# 10b — Teste do cache do SVM e do paralelismo de treino (complemento da calibração)

**Por que este notebook existe.** Na calibração (notebook 10), o treino com 40 mil linhas levou **6,6 s**, mas com 81 mil linhas levou **341 s** (52× mais para 2× as linhas; a tendência dos tamanhos menores previa ~26 s). O processo ficou calculando (tempo de CPU ≈ relógio), então não foi memória virtual nem disco.

**Hipóteses (nenhuma confirmada).**
1. **Cache pequeno.** O SVM guarda em um cache as linhas da matriz de kernel de que precisa (padrão: 200 MiB). Cada linha tem 4 bytes × (linhas de treino): com 81 mil linhas, o cache comporta só ~645 linhas, e o algoritmo usa ~2.152 (os vetores de suporte). Quando o cache é menor que isso, as linhas são recalculadas toda hora. **No meu teste sintético o efeito foi de ~2× (com um cache minúsculo); nos seus dados pode ser maior ou não ser a causa.**
2. **O computador estava sob pressão** (só ~0,8 a 1,3 GiB de RAM livre e o Meet aberto): a medição pode não se repetir com a máquina mais livre.
3. **A lentidão é do próprio problema** (muito mais iterações do algoritmo com mais linhas), e nesse caso o tempo de 341 s é o real.

| O que este notebook faz | Como |
|---|---|
| **Teste do cache** | Treina o SVM com o ajuste inteiro e vários tamanhos de cache (do maior para o menor). **O resultado do SVM não depende do cache** (mesmos vetores de suporte e mesmo ROC-AUC): só o tempo muda. O notebook compara com o tempo que a tendência dos tamanhos menores previa e **separa as três hipóteses** (cache, ambiente ou problema) |
| **Paralelismo de treino** | Na calibração ele ficou **sem medir** (pouca RAM livre). Repete o teste com a memória disponível agora e o cache recomendado |
| **Estimativas atualizadas** | Refaz o tempo da busca, do refit e da previsão do teste com o tempo medido com o cache recomendado |

## Proteções
Cada treino roda num processo à parte, com limite de tempo (`ORCAMENTO_FIT_S`, padrão 20 min). Um tamanho de cache que **não cabe na RAM livre** é pulado (com aviso). A espera é interrompível.

## Antes de rodar
**Libere memória.** Na calibração havia só ~0,8 a 1,3 GiB livres (de 7,7 GiB). Para testar caches de 1.000 a 1.500 MiB são necessários uns 2 a 3 GiB livres: **feche o Meet, o Edge com abas pesadas e os outros notebooks**. Se a memória não bastar, o notebook pula os caches grandes e avisa. O caso de 200 MiB (o de controle) deve levar uns 6 minutos. Tempo total estimado: ~10 a 25 minutos.

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
def _div(a, b): return float(a) / float(b) if b else float("nan")

def metricas(y, p, limiar=0.5, w=None):
    # Todas as métricas de uma vez. Com `w` (pesos), as contagens e as métricas que dependem da proporção ficam ponderadas.
    # O ROC-AUC é calculado sem pesos: ele só compara ataques com BENIGN e não depende da proporção.
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float); pred = (p >= limiar).astype(int)
    w_ = np.ones(len(y)) if w is None else np.asarray(w, dtype=float)
    tp = float(w_[(y == 1) & (pred == 1)].sum()); tn = float(w_[(y == 0) & (pred == 0)].sum())
    fp = float(w_[(y == 0) & (pred == 1)].sum()); fn = float(w_[(y == 1) & (pred == 0)].sum())
    rec, esp, prec = _div(tp, tp + fn), _div(tn, tn + fp), _div(tp, tp + fp)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    f1 = _div(2 * prec * rec, prec + rec) if (prec == prec and rec == rec) else float("nan")
    prec_b = _div(tn, tn + fn); f1_b = _div(2 * prec_b * esp, prec_b + esp) if (prec_b == prec_b and esp == esp) else float("nan")    # classe BENIGN como "positiva"
    return {"acuracia": _div(tp + tn, tp + tn + fp + fn), "acuracia_balanceada": (rec + esp) / 2, "precisao": prec, "recall": rec, "especificidade": esp,
            "taxa_falso_alarme": 1 - esp, "taxa_perda": 1 - rec, "f1": f1, "mcc": _div(tp * tn - fp * fn, den),
            "precisao_benign": prec_b, "f1_benign": f1_b, "f1_macro": (f1 + f1_b) / 2, "pr_auc_benign": float(average_precision_score(1 - y, 1 - p, sample_weight=w_)),
            "roc_auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p, sample_weight=w_)), "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def recall_em_fpr(y, p, alvo):
    # Ponto de operação: o MAIOR recall com taxa de falso alarme <= alvo (limiar escolhido nestes dados).
    fpr, tpr, thr = roc_curve(np.asarray(y).astype(int), np.asarray(p, dtype=float))
    i = max(int(np.searchsorted(fpr, alvo, side="right")) - 1, 0)
    lim = float(thr[i]) if np.isfinite(thr[i]) else 1.0 + 1e-9
    return {"alvo_fpr": alvo, "limiar": lim, "recall": float(tpr[i]), "fpr": float(fpr[i])}

class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# autoteste das ferramentas (não toca no servidor)
_y = np.array([0, 0, 0, 0, 1, 1, 1, 1]); _p = np.array([.1, .2, .3, .6, .4, .7, .8, .9]); _m = metricas(_y, _p, 0.5)
assert (_m["tp"], _m["fp"], _m["tn"], _m["fn"]) == (3, 1, 3, 1) and abs(_m["mcc"] - 0.5) < 1e-12 and abs(_m["acuracia"] - 0.75) < 1e-12, _m
_o = recall_em_fpr(_y, _p, 0.25); assert abs(_o["recall"] - 1.0) < 1e-12 and _o["fpr"] <= 0.25 and abs(_o["limiar"] - 0.4) < 1e-12, _o
_m2 = metricas(_y, _p, 0.5); assert abs(_m2["precisao_benign"] - 0.75) < 1e-12 and abs(_m2["f1_benign"] - 0.75) < 1e-12 and abs(_m2["f1_macro"] - 0.75) < 1e-12, _m2
_o0 = recall_em_fpr(_y, _p, 0.0); assert abs(_o0["recall"] - 0.75) < 1e-12 and _o0["fpr"] == 0.0, _o0
_d = distribuir(100, {"a": 1000, "b": 1000, "c": 1000, "d": 10}, 0.0, {"a": 1000, "b": 1000, "c": 1000, "d": 10}); assert _d["d"] == 10 and sum(_d.values()) == 100, _d
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


### Barra de progresso
`Barra` mostra o andamento (percentual, tempo decorrido e tempo restante estimado) das etapas com várias medições.

In [6]:
import gc

def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")

_b = Barra(2, "autoteste"); _b.avancar(2); assert fmt_tempo(3725) == "1h 02min"
print("Barra de progresso pronta")

Barra de progresso pronta


## 1. Parâmetros, dados e normalizador
Lê o ajuste e a validação (as mesmas views do RF, ordenadas por `id_linha`), junta o tipo de ataque (para as amostras serem **estratificadas por tipo**) e define o `Pipeline` do SVM. `novo_pipeline(C, gamma, escala)` cria o modelo: `escala = "padrao"` (o escolhido), `"robusto"` (mediana e intervalo interquartil, menos sensível a valores extremos) ou `"nenhum"` (só para comparar).

In [7]:
# ---------- 1.1 Parâmetros, dados e normalizador ----------
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
from joblib import Parallel, delayed
from joblib.externals.loky import ProcessPoolExecutor as LokyPool
import concurrent.futures

TAMANHOS = tuple(globals().get("TAMANHOS", (10_000, 20_000, 40_000, None)))          # None = o ajuste inteiro
PROCESSOS = tuple(globals().get("PROCESSOS", (1, 2, 4, 6, 8)))                       # processos simultâneos a testar
C_VALORES = tuple(globals().get("C_VALORES", (0.1, 1.0, 10.0, 100.0)))
GAMMA_FATORES = tuple(globals().get("GAMMA_FATORES", (0.25, 1.0, 4.0)))              # múltiplos do gamma "scale"
CACHE_MB = globals().get("CACHE_MB", 200)                                            # cache de kernel de cada SVM (por processo)
ORCAMENTO_FIT_S = globals().get("ORCAMENTO_FIT_S", 1200)                             # nenhum treino pode passar disto (o processo é encerrado)
ORCAMENTO_SENS_S = globals().get("ORCAMENTO_SENS_S", 90)                             # as comparações usam o maior tamanho cujo treino levou até isto
LIMITE_BUSCA_H = globals().get("LIMITE_BUSCA_H", 6.0); LIMITE_TESTE_H = globals().get("LIMITE_TESTE_H", 1.5)    # para a recomendação (em horas)
SEMENTE = globals().get("SEMENTE", 42)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PASTA = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "svm_binario_calibracao" / f"svm_cache_{RUN_ID}"; PASTA.mkdir(parents=True, exist_ok=True)

V_AJUSTE, V_VALID, V_REFIT, SEL_TAB = "prep_vw_bal_bin_treino", "prep_vw_bal_bin_validacao", "prep_vw_bal_bin_refit", "prep_bal_bin_selecao"
VIEW_TESTE = "prep_vw_rf_bin_teste_sem_sobrep"
faltam = [t for t in ("prep_final_treino", V_AJUSTE, V_VALID, V_REFIT, SEL_TAB, VIEW_TESTE, "prep_views_rf_binario") if not existe(t)]
assert not faltam, f"Rode antes os notebooks 02 a 05: faltam {faltam}"
aud = consultar_num("SELECT posicao, feature, situacao, mediana_imputacao, tem_indicador, versao FROM prep_views_rf_binario ORDER BY posicao"); versao_views = str(aud["versao"].iloc[0])
USADAS, RETIRADAS = aud[aud.situacao == "usada"], aud[aud.situacao != "usada"].feature.tolist()
MEDIANAS = {r.feature: float(r.mediana_imputacao) for r in USADAS.itertuples() if int(r.tem_indicador) == 1}
prep = PreprocessadorRF(USADAS.feature.tolist(), MEDIANAS, RETIRADAS, versao_views); CM = prep.colunas_modelo
assert CM == [c for c in colunas_view(V_AJUSTE) if c not in ("id_linha", "y")], "a ordem das colunas da view difere da do pré-processamento"
NUM, IND = list(range(len(prep.features))), list(range(len(prep.features), len(CM)))           # as colunas numéricas vêm primeiro; os indicadores _ausente, no fim

def ordenar(df): return df.iloc[np.argsort(df["id_linha"].astype("uint64").to_numpy(), kind="stable")].reset_index(drop=True)
aj, va = ordenar(ler_view(V_AJUSTE)), ordenar(ler_view(V_VALID))
X_aj, y_aj, X_va, y_va = aj[CM].to_numpy(float), aj["y"].to_numpy(int), va[CM].to_numpy(float), va["y"].to_numpy(int)
for nome, X in (("ajuste", X_aj), ("validação", X_va)): assert np.isfinite(X).all(), f"valores não finitos em {nome}"
nome_tipo = lambda t: t.replace("DrDoS_", "", 1) if t.startswith("DrDoS_") else t
sel = consultar(f"SELECT toString(id_linha) AS id_linha, papel, tipo FROM {SEL_TAB}"); sel["tipo"] = sel["tipo"].map(nome_tipo)
tipo_aj = aj[["id_linha"]].merge(sel[sel.papel == "treino"][["id_linha", "tipo"]], on="id_linha", how="left")["tipo"]; assert tipo_aj.notna().all()
g_aj = tipo_aj.astype("category").cat.codes.to_numpy()
N_AJ, N_VA, N_RF, N_TESTE = len(X_aj), len(X_va), n_linhas(V_REFIT), n_linhas(VIEW_TESTE)

def novo_pipeline(C=1.0, gamma="scale", escala="padrao", cache=None):
    # Pipeline do SVM: normaliza (média e desvio do próprio treino) só as colunas numéricas; os indicadores _ausente ficam 0/1; depois o SVM RBF.
    if escala == "nenhum": pre = "passthrough"
    else:
        esc = StandardScaler() if escala == "padrao" else RobustScaler()
        pre = ColumnTransformer([("num", esc, NUM)] + ([("ind", "passthrough", IND)] if IND else []))
    return Pipeline([("escala", pre), ("svm", SVC(kernel="rbf", C=C, gamma=gamma, cache_size=cache or CACHE_MB))])

def amostra(n):
    # n linhas do ajuste, estratificadas por tipo (BENIGN continua 50%); None = tudo
    if n is None or n >= N_AJ: return np.arange(N_AJ)
    idx, _ = train_test_split(np.arange(N_AJ), train_size=n, stratify=g_aj, random_state=SEMENTE); return np.sort(idx)

print(f"Pasta da calibração: {PASTA}")
print(f"Ajuste: {N_AJ:,} linhas ({int((y_aj == 0).sum()):,} BENIGN) | validação: {N_VA:,} ({int((y_va == 0).sum()):,} BENIGN) | refit: {N_RF:,} | teste: {N_TESTE:,}")
print(f"Atributos: {len(CM)} = {len(NUM)} numéricos (padronizados) + {len(IND)} indicadores _ausente (ficam 0/1)")
print(f"Computador: {os.cpu_count()} núcleos lógicos" + (f" ({psutil.cpu_count(logical=False)} físicos)" if psutil else "") + (f", {psutil.virtual_memory().total / 2 ** 30:.1f} GiB de RAM, {psutil.virtual_memory().available / 2 ** 30:.1f} GiB livres agora" if psutil else ""))
# conferência do normalizador: ajustado só no treino, numéricos com média 0 e desvio 1, indicadores intactos
_p = novo_pipeline().fit(X_aj[amostra(5000)], y_aj[amostra(5000)]); _Xt = _p.named_steps["escala"].transform(X_aj[amostra(5000)])
assert np.allclose(_Xt[:, NUM].mean(0), 0, atol=1e-9) and np.allclose(_Xt[:, NUM].std(0), 1, atol=1e-6) or (_Xt[:, NUM].std(0) == 0).any(), "o normalizador não deixou as colunas numéricas com média 0 e desvio 1"
assert (not IND) or np.array_equal(_Xt[:, IND], X_aj[amostra(5000)][:, IND]), "os indicadores _ausente foram alterados pelo normalizador"
print("✔ Normalizador conferido: colunas numéricas com média 0 e desvio 1 no treino; indicadores _ausente intactos (0/1)")

Pasta da calibração: C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_binario_calibracao\svm_cache_20261024_233500
Ajuste: 81,228 linhas (40,614 BENIGN) | validação: 60,154 (10,154 BENIGN) | refit: 101,536 | teste: 7,219,189
Atributos: 43 = 42 numéricos (padronizados) + 1 indicadores _ausente (ficam 0/1)
Computador: 12 núcleos lógicos (10 físicos), 7.7 GiB de RAM, 1.2 GiB livres agora
✔ Normalizador conferido: colunas numéricas com média 0 e desvio 1 no treino; indicadores _ausente intactos (0/1)


## 2. Ferramentas da calibração
- **`_fit_worker`**: treina um SVM (`Pipeline`) numa amostra e, no mesmo processo, mede tempo de relógio e de CPU, pico de RAM, vetores de suporte, o tempo de transformar e de prever a validação (daí o **custo por par**, em ns) e, de forma informativa, ROC-AUC, recall e falso alarme da validação (limiar 0 do SVM).
- **`executar_com_limite`**: roda uma função **num processo separado com limite de tempo**; se passar, o processo é encerrado e o notebook segue.
- **`MedidorSistema`**: mede quanto a RAM livre do computador caiu durante um bloco (soma de todos os processos).
- **`ajustar_potencia`**: ajusta `tempo = a × n^b` (e o mesmo para os vetores de suporte), para estimar tamanhos que não foram medidos.

In [8]:
class MedidorSistema:
    # Quanto a RAM livre do computador (todos os programas) caiu durante um bloco `with`.
    def __init__(self, intervalo=0.2): self.intervalo = intervalo
    def __enter__(self):
        self.inicio = psutil.virtual_memory().available if psutil else float("nan"); self.minimo = self.inicio
        if psutil:
            self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set(): self.minimo = min(self.minimo, psutil.virtual_memory().available); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        if psutil: self._parar.set(); self._th.join(); self.minimo = min(self.minimo, psutil.virtual_memory().available)
        self.livre_inicial_gib = self.inicio / 2 ** 30; self.queda_gib = (self.inicio - self.minimo) / 2 ** 30

def _fit_worker(X, y, Xv, yv, C, gamma, escala, devolver_modelo=False, cache=None):
    pipe = novo_pipeline(C, gamma, escala, cache)
    with Medidor() as m: pipe.fit(X, y)
    svm, pre = pipe.named_steps["svm"], pipe.named_steps["escala"]
    t0 = time.perf_counter(); Xt = Xv if isinstance(pre, str) else pre.transform(Xv); t_tr = time.perf_counter() - t0
    t0 = time.perf_counter(); sc = svm.decision_function(Xt); t_sv = time.perf_counter() - t0
    nsv = int(svm.n_support_.sum())
    r = {"n": int(len(y)), "C": float(C), "gamma": float(getattr(svm, "_gamma", float("nan"))), "escala": escala, "cache_mb": cache or CACHE_MB, "tempo_treino_s": m.relogio, "cpu_s": m.cpu, "pico_ram_mib": m.pico_mib, "acrescimo_ram_mib": m.delta_mib,
         "n_sv": nsv, "frac_sv": nsv / len(y), "tempo_transformacao_val_s": t_tr, "tempo_svm_val_s": t_sv, "ns_por_par": t_sv / (len(Xv) * max(nsv, 1)) * 1e9,
         "roc_auc_val": float(roc_auc_score(yv, sc)), "recall_0": float((sc[yv == 1] > 0).mean()), "fpr_0": float((sc[yv == 0] > 0).mean())}
    return (r, pipe if devolver_modelo else None)

def executar_com_limite(fn, args, limite_s, barra=None, rotulo=""):
    # Um processo NOVO e independente por chamada (não compartilha nada com o Parallel do joblib); se passar do limite, é encerrado.
    # Espera em fatias de 0,5 s: o botão "interromper" do Jupyter funciona (também no Windows) e a barra mostra o tempo decorrido.
    ex = LokyPool(max_workers=1); fut = ex.submit(fn, *args); t0 = time.perf_counter()
    try:
        while True:
            decorrido = time.perf_counter() - t0
            if decorrido >= limite_s: ex.shutdown(wait=True, kill_workers=True); return None
            if barra is not None: barra._extra = f"{rotulo} | em andamento há {fmt_tempo(decorrido)} (limite {fmt_tempo(limite_s)})"; barra._mostrar()
            try: res = fut.result(timeout=min(0.5, limite_s - decorrido))
            except (concurrent.futures.TimeoutError, TimeoutError): continue
            ex.shutdown(wait=True); return res
    except BaseException:
        ex.shutdown(wait=True, kill_workers=True); raise

def ajustar_potencia(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float); ok_ = (x > 0) & (y > 0); b, loga = np.polyfit(np.log(x[ok_]), np.log(y[ok_]), 1)
    return (lambda n: float(np.exp(loga) * float(n) ** b)), float(b)
print("Ferramentas da calibração prontas")

Ferramentas da calibração prontas


## 2. Cache × tempo
Treina o SVM (`C = 1`, `gamma = "scale"`, normalizador padrão) no tamanho que ficou lento, com vários tamanhos de cache, **do maior para o menor** (a resposta chega mais cedo; o de 200 MiB, o de controle, vai por último). A coluna `linhas_no_cache / vetores_de_suporte` mostra se o cache comporta o que o algoritmo precisa (abaixo de 1 = não comporta). O ROC-AUC é só informativo, e o número de vetores de suporte deve ser **igual** em todos os caches.

**Se nenhum tamanho de cache couber na memória livre, o notebook PARA aqui com uma mensagem** (e mostra quais programas mais ocupam a memória), em vez de seguir adiante sem dados. O cache de 200 MiB precisa de ~0,6 GiB livres; os de 1.000 a 1.500 MiB, de 1,4 a 1,9 GiB.

In [9]:
# ---------- 2.1 Cache × tempo ----------
CACHES_MB = tuple(globals().get("CACHES_MB", (1500, 1000, 500, 200)))
N_CACHE = globals().get("N_CACHE", None)                              # None = o ajuste inteiro (o tamanho que ficou lento)
LIM_C = globals().get("ORCAMENTO_CACHE_S", ORCAMENTO_FIT_S)
n_c = N_AJ if N_CACHE is None else min(N_CACHE, N_AJ); idx_c = amostra(n_c); Xc, yc = X_aj[idx_c], y_aj[idx_c]
print(f"Tamanho testado: {n_c:,} linhas | cada linha do cache ocupa {4 * n_c / 2 ** 10:,.0f} KiB: um cache de 200 MiB guarda só {200 * 2 ** 20 / (4 * n_c):,.0f} linhas")
def top_memoria(n=7):
    # quanto cada programa ocupa de RAM agora (soma dos processos de mesmo nome), para você saber o que fechar
    soma = {}
    for p in psutil.process_iter(["name", "memory_info"]):
        try: soma[p.info["name"]] = soma.get(p.info["name"], 0) + p.info["memory_info"].rss
        except Exception: pass
    return sorted(soma.items(), key=lambda kv: -kv[1])[:n]
if psutil:
    print(f"RAM livre agora: {psutil.virtual_memory().available / 2 ** 30:.1f} GiB de {psutil.virtual_memory().total / 2 ** 30:.1f} GiB | cada tamanho de cache precisa de ~(cache + 400 MiB) livres")
    print("Programas que mais ocupam memória agora: " + "; ".join(f"{nome} {b / 2 ** 30:.1f} GiB" for nome, b in top_memoria()))
linhas_c = []; barra = Barra(len(CACHES_MB), "Cache do SVM")
for cmb in CACHES_MB:
    livre = psutil.virtual_memory().available / 2 ** 30 if psutil else float("inf"); precisa = (cmb + 400) / 1024                    # cache + processo do treino (dados, modelo e bibliotecas, ~400 MiB medidos)
    if precisa > 0.9 * livre:
        print(f"  ⚠ o cache de {cmb} MiB precisaria de ~{precisa:.1f} GiB livres e há {livre:.1f} GiB: pulado (feche programas e rode de novo)"); linhas_c.append({"cache_mb": cmb, "concluiu": False, "motivo": "pouca memória"}); barra.avancar(1, f"cache {cmb} MiB: pulado (pouca memória)"); continue
    barra.avancar(0, f"cache {cmb} MiB...")
    out = executar_com_limite(_fit_worker, (Xc, yc, X_va, y_va, 1.0, "scale", "padrao", False, cmb), LIM_C, barra, f"cache de {cmb} MiB")
    if out is None:
        linhas_c.append({"cache_mb": cmb, "concluiu": False, "motivo": f"passou de {fmt_tempo(LIM_C)}"}); barra.avancar(1, f"cache {cmb} MiB: PASSOU DO LIMITE e foi encerrado"); continue
    r = out[0]; linhas_c.append({"cache_mb": cmb, "concluiu": True, "motivo": "", **{k: r[k] for k in ("n", "tempo_treino_s", "cpu_s", "pico_ram_mib", "n_sv", "roc_auc_val", "ns_por_par")}})
    barra.avancar(1, f"cache {cmb} MiB: {fmt_tempo(r['tempo_treino_s'])}, {r['n_sv']:,} vetores de suporte")
barra.fechar(); tab_cache = pd.DataFrame(linhas_c); tab_cache["linhas_no_cache"] = tab_cache.cache_mb * 2 ** 20 / (4 * n_c); tab_cache["linhas_no_cache / vetores_de_suporte"] = tab_cache.linhas_no_cache / tab_cache.get("n_sv", np.nan)
print(f"\n== Cache × tempo de treino ({n_c:,} linhas) =="); display(tab_cache.round(3))
ok_c = tab_cache[tab_cache.concluiu]
if not len(ok_c):
    motivos = tab_cache["motivo"].value_counts().to_dict(); falta_mem = "pouca memória" in motivos
    raise RuntimeError("Nenhum tamanho de cache foi concluído (" + "; ".join(f"{v}× {k}" for k, v in motivos.items()) + "). " + ("FALTA MEMÓRIA LIVRE: feche o Meet, as abas do Edge e os outros notebooks (veja acima quais programas mais ocupam memória; Gerenciador de Tarefas → Desempenho → Memória), "
                       "reinicie o kernel se preciso e rode de novo a partir da primeira célula. Com menos de ~0,7 GiB livres nem o cache de 200 MiB cabe. " if falta_mem else "") + ("Os treinos passaram do limite de tempo: aumente ORCAMENTO_CACHE_S." if any("passou" in str(k) for k in motivos) else ""))
CACHE_REC = 200; T_REF = float("nan"); NSV_REF = float("nan")
if len(ok_c):
    tmin = ok_c.tempo_treino_s.min(); CACHE_REC = int(ok_c[ok_c.tempo_treino_s <= 1.25 * tmin].cache_mb.min()); lin = ok_c[ok_c.cache_mb == CACHE_REC].iloc[0]; T_REF, NSV_REF, PICO_REF = float(lin.tempo_treino_s), float(lin.n_sv), float(lin.pico_ram_mib)
    ctrl = ok_c[ok_c.cache_mb == 200]
    cal_t = sorted(PASTA.parent.glob("svm_cal_*/calibracao_tamanhos.csv")); T_CAL = T_ESP = float("nan")
    if cal_t:
        ct = pd.read_csv(cal_t[-1]).sort_values("n"); igual = ct[ct.n == n_c]; menor = ct[ct.n < n_c]
        if len(igual): T_CAL = float(igual.tempo_treino_s.iloc[0])
        if len(menor): T_ESP = float(menor.tempo_treino_s.iloc[-1]) * (n_c / float(menor.n.iloc[-1])) ** 1.95                  # o que a tendência dos tamanhos menores previa
    print(f"\nReferência da calibração: treino com {n_c:,} linhas levou {fmt_tempo(T_CAL)}; a tendência dos tamanhos menores previa ~{fmt_tempo(T_ESP)}.")
    if len(ctrl) and CACHE_REC != 200 and ctrl.tempo_treino_s.iloc[0] / T_REF >= 2: print(f"→ VEREDITO: CACHE. Com {CACHE_REC} MiB o treino levou {fmt_tempo(T_REF)}; com 200 MiB, {fmt_tempo(ctrl.tempo_treino_s.iloc[0])} ({ctrl.tempo_treino_s.iloc[0] / T_REF:.1f}× mais). O cache pequeno era o gargalo: use o cache recomendado na busca.")
    elif T_ESP == T_ESP and ok_c.tempo_treino_s.max() <= 2.5 * T_ESP: print(f"→ VEREDITO: AMBIENTE. Todos os tamanhos de cache ficaram perto do esperado (~{fmt_tempo(T_ESP)}), inclusive o de 200 MiB: a lentidão da calibração NÃO se repetiu. A causa mais provável é a máquina sob pressão naquela hora (pouca RAM livre, Meet ou outro programa). Rode a busca com a máquina livre.")
    elif T_CAL == T_CAL and ok_c.tempo_treino_s.min() >= 0.4 * T_CAL: print(f"→ VEREDITO: PROBLEMA. O treino continua lento (mínimo {fmt_tempo(ok_c.tempo_treino_s.min())}) mesmo com cache grande e a máquina livre: a lentidão é do próprio problema (muitas iterações). As estimativas devem usar o tempo medido, e a busca precisa de amostra ou de um plano mais leve.")
    else: print("→ VEREDITO: inconclusivo. Compare a tabela acima com a referência e me envie os números.")
    if len(ok_c) == 1: print(f"(só um tamanho de cache foi medido: {CACHE_REC} MiB; rode de novo com mais memória livre para comparar)")
    print(f"→ cache recomendado: {CACHE_REC} MiB (o menor que ficou a até 25% do tempo mínimo)")
else: print("⚠ nenhum cache concluiu: libere memória e rode de novo.")

Tamanho testado: 81,228 linhas | cada linha do cache ocupa 317 KiB: um cache de 200 MiB guarda só 645 linhas
RAM livre agora: 1.1 GiB de 7.7 GiB | cada tamanho de cache precisa de ~(cache + 400 MiB) livres
Programas que mais ocupam memória agora: msedge.exe 1.6 GiB; svchost.exe 0.9 GiB; python3.13.exe 0.4 GiB; MemCompression 0.2 GiB; explorer.exe 0.2 GiB; msedgewebview2.exe 0.1 GiB; RuntimeBroker.exe 0.1 GiB


  ⚠ o cache de 1500 MiB precisaria de ~1.9 GiB livres e há 1.2 GiB: pulado (feche programas e rode de novo)
  ⚠ o cache de 1000 MiB precisaria de ~1.4 GiB livres e há 1.2 GiB: pulado (feche programas e rode de novo)
Cache do SVM: concluído em 9min 50s

== Cache × tempo de treino (81,228 linhas) ==


,cache_mb,concluiu,motivo,n,tempo_treino_s,cpu_s,pico_ram_mib,n_sv,roc_auc_val,ns_por_par,linhas_no_cache,linhas_no_cache / vetores_de_suporte
0,1500,False,pouca memória,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4840.892,NaN
1,1000,False,pouca memória,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3227.262,NaN
2,500,True,,81228.0,274.783,271.453,834.523,2152.0,0.999,126.674,1613.631,0.75
3,200,True,,81228.0,264.194,262.484,494.715,2152.0,0.999,223.477,645.452,0.30



Referência da calibração: treino com 81,228 linhas levou 5min 41s; a tendência dos tamanhos menores previa ~26s.
→ VEREDITO: PROBLEMA. O treino continua lento (mínimo 4min 24s) mesmo com cache grande e a máquina livre: a lentidão é do próprio problema (muitas iterações). As estimativas devem usar o tempo medido, e a busca precisa de amostra ou de um plano mais leve.
→ cache recomendado: 200 MiB (o menor que ficou a até 25% do tempo mínimo)


## 3. Paralelismo de treino (agora com o cache recomendado)
Repete o teste que ficou sem medir: vários treinos ao mesmo tempo, cada um com o **cache recomendado**, e a RAM livre que isso consome. Um número de processos que **não cabe na memória livre** é pulado. Recomenda mais de 1 processo só se o ganho for de pelo menos 1,2× com eficiência ≥ 0,5.

In [10]:
# ---------- 3.1 Treinos simultâneos ----------
PROCESSOS = tuple(globals().get("PROCESSOS_TREINO", (1, 2, 3, 4))); N_PAR = globals().get("N_PAR", 40_000); idx_p = amostra(N_PAR); Xp, yp = X_aj[idx_p], y_aj[idx_p]; Xvp, yvp = X_va[:2000], y_va[:2000]; assert len(np.unique(yvp)) == 2
IGNORAR_NUCLEOS = globals().get("IGNORAR_NUCLEOS", False)                                         # só para testes
procs = sorted({1} | {k for k in PROCESSOS if IGNORAR_NUCLEOS or k <= (os.cpu_count() or 1)}); por_proc_gib = ((PICO_REF if len(ok_c) else 600) + CACHE_REC) * 1.2 / 1024; linhas_par = []; barra = Barra(len(procs), f"Treinos simultâneos ({N_PAR:,} linhas, cache {CACHE_REC} MiB)")
print(f"Cada processo precisa de ~{por_proc_gib:.1f} GiB (pico medido + cache); RAM livre agora: " + (f"{psutil.virtual_memory().available / 2 ** 30:.1f} GiB" if psutil else "?"))
for k in procs:
    if psutil and k > 1 and k * por_proc_gib > 0.8 * psutil.virtual_memory().available / 2 ** 30:
        print(f"  ⚠ {k} processos precisariam de ~{k * por_proc_gib:.1f} GiB e só há {psutil.virtual_memory().available / 2 ** 30:.1f} GiB livres: pulado"); barra.avancar(1, f"{k} processos: pulado (pouca memória)"); continue
    if k > 1: Parallel(n_jobs=k)(delayed(time.sleep)(0.01) for _ in range(k))
    barra.avancar(0, f"{k} treino(s) ao mesmo tempo...")
    with MedidorSistema() as ms:
        t0 = time.perf_counter(); outs = Parallel(n_jobs=k)(delayed(_fit_worker)(Xp, yp, Xvp, yvp, 1.0, "scale", "padrao", False, CACHE_REC) for _ in range(k)); wall = time.perf_counter() - t0
    rs = [o[0] for o in outs]; linhas_par.append({"processos": k, "tempo_total_s": wall, "tempo_medio_por_treino_s": float(np.mean([r["tempo_treino_s"] for r in rs])), "n_sv_igual_em_todos": len({r["n_sv"] for r in rs}) == 1, "n_sv": rs[0]["n_sv"],
                                                    "ram_livre_inicial_gib": ms.livre_inicial_gib, "queda_ram_gib": ms.queda_gib}); barra.avancar(1, f"{k} processo(s): {fmt_tempo(wall)} no total")
barra.fechar(); tab_par = pd.DataFrame(linhas_par); v1 = 1 / float(tab_par.loc[tab_par.processos == 1, "tempo_total_s"].iloc[0])
tab_par["treinos_por_min"] = 60 * tab_par.processos / tab_par.tempo_total_s; tab_par["ganho"] = (tab_par.processos / tab_par.tempo_total_s) / v1; tab_par["eficiencia"] = tab_par.ganho / tab_par.processos
bons = tab_par[(tab_par.queda_ram_gib <= 0.7 * tab_par.ram_livre_inicial_gib) & (tab_par.eficiencia >= 0.5)]; K_REC = int(bons.loc[bons.ganho.idxmax(), "processos"]) if (len(bons) and bons.ganho.max() >= 1.2) else 1
GANHO_TREINO = {int(r.processos): float(r.ganho) for r in tab_par.itertuples()}
print(f"\n== Treinos simultâneos ({N_PAR:,} linhas cada, cache de {CACHE_REC} MiB) =="); display(tab_par.round(3)); print(f"→ número recomendado de treinos simultâneos: {K_REC} (ganho {GANHO_TREINO[K_REC]:.1f}×)")

Cada processo precisa de ~0.8 GiB (pico medido + cache); RAM livre agora: 0.4 GiB
  ⚠ 2 processos precisariam de ~1.6 GiB e só há 1.1 GiB livres: pulado
  ⚠ 3 processos precisariam de ~2.4 GiB e só há 1.1 GiB livres: pulado
  ⚠ 4 processos precisariam de ~3.3 GiB e só há 1.1 GiB livres: pulado
Treinos simultâneos (40,000 linhas, cache 200 MiB): concluído em 12s

== Treinos simultâneos (40,000 linhas cada, cache de 200 MiB) ==


,processos,tempo_total_s,tempo_medio_por_treino_s,n_sv_igual_em_todos,n_sv,ram_livre_inicial_gib,queda_ram_gib,treinos_por_min,ganho,eficiencia
0,1,11.855,11.374,True,1252,0.365,0.016,5.061,1.0,1.0


→ número recomendado de treinos simultâneos: 1 (ganho 1.0×)


## 4. Estimativas atualizadas
Refaz as estimativas do notebook 10 a partir do tempo **medido** com o cache recomendado no tamanho do ajuste. Os demais tamanhos são **aproximações**: o tempo escala com `n^1,5` (expoente **suposto**, `EXPOENTE_TEMPO`; para SVMs com poucos vetores de suporte e cache suficiente, é uma ordem de grandeza razoável, mas confirme na primeira execução real). O custo por par, os vetores de suporte e o ganho da previsão em paralelo vêm da calibração anterior (`calibracao_svm.json`).

In [11]:
# ---------- 4.1 Estimativas atualizadas ----------
cals = sorted(PASTA.parent.glob("svm_cal_*/calibracao_svm.json")); assert cals, "não achei o calibracao_svm.json do notebook 10: rode antes o notebook 10"
ref = json.loads(cals[-1].read_text(encoding="utf-8")); NS_PAR = float(ref["medidas"]["ns_por_par"]); B_SV = float(ref["medidas"]["expoente_vetores_suporte"]); GANHO_PREV = {int(k): float(v) for k, v in ref["medidas"]["ganho_previsao"].items()}; K_PREV = int(ref["medidas"]["processos_recomendados_previsao"])
EXPOENTE_TEMPO = globals().get("EXPOENTE_TEMPO", 1.5); S_T = GANHO_TREINO.get(K_REC, 1.0); S_P = GANHO_PREV.get(K_PREV, 1.0)
f_t = lambda n: T_REF * (n / n_c) ** EXPOENTE_TEMPO; f_sv = lambda n: NSV_REF * (n / n_c) ** B_SV; prev_s = lambda linhas, n_sv: linhas * n_sv * NS_PAR * 1e-9
proc_txt = lambda k: f"{k} processo" + ("" if k == 1 else "s")
I = {"busca_val": ("Busca com validação separada (9 treinos)", 9 * (f_t(N_AJ) + prev_s(N_VA, f_sv(N_AJ))), True), "busca_cv": ("Busca com CV de 5 dobras (45 treinos)", 45 * (f_t(0.8 * N_AJ) + prev_s(0.2 * N_AJ, f_sv(0.8 * N_AJ))), True),
     "refit": ("Refit (1 treino)", f_t(N_RF), False), "teste": (f"Previsão do teste ({N_TESTE:,} linhas)", prev_s(N_TESTE, f_sv(N_RF)), False)}
linhas_e = []
for k, (nome, seq, paral) in I.items():
    par = seq / S_T if paral else (seq / S_P if k == "teste" else seq); linhas_e.append({"chave": k, "item": nome, "1_processo_s": seq, "paralelo_s": par, "1 processo": fmt_tempo(seq), proc_txt(K_REC if paral else (K_PREV if k == "teste" else 1)): fmt_tempo(par)})
tab_est2 = pd.DataFrame(linhas_e); print(f"== Estimativas atualizadas (cache {CACHE_REC} MiB; tempo medido em {n_c:,} linhas = {fmt_tempo(T_REF)}; expoente suposto {EXPOENTE_TEMPO}; ganho do treino {S_T:.1f}× com {proc_txt(K_REC)}; ganho da previsão {S_P:.1f}× com {proc_txt(K_PREV)}) ==")
display(tab_est2.drop(columns=["1_processo_s", "paralelo_s"]).fillna("")); E2 = {r.chave: r.paralelo_s / 3600 for r in tab_est2.itertuples()}
LIM_B, LIM_T = ref["limites_h"]["busca"], ref["limites_h"]["teste"]
if E2["busca_cv"] <= LIM_B: print(f"✔ SVM binário: a CV de 5 dobras cabe (~{fmt_tempo(tab_est2.set_index('chave').loc['busca_cv', 'paralelo_s'])}).")
elif E2["busca_val"] <= LIM_B: print(f"⚠ SVM binário: a CV completa não cabe; use a validação separada (~{fmt_tempo(tab_est2.set_index('chave').loc['busca_val', 'paralelo_s'])}) + CV nas finalistas.")
else: print("✘ SVM binário: nem a validação separada cabe: busca em amostra e retreino do melhor.")
print(("✔" if E2["teste"] <= LIM_T else "⚠") + f" Previsão do teste: ~{fmt_tempo(tab_est2.set_index('chave').loc['teste', 'paralelo_s'])}")

== Estimativas atualizadas (cache 200 MiB; tempo medido em 81,228 linhas = 4min 24s; expoente suposto 1.5; ganho do treino 1.0× com 1 processo; ganho da previsão 1.8× com 2 processos) ==


,chave,item,1 processo,2 processos
0,busca_val,Busca com validação separada (9 treinos),43min 33s,
1,busca_cv,Busca com CV de 5 dobras (45 treinos),2h 26min,
2,refit,Refit (1 treino),6min 09s,
3,teste,"Previsão do teste (7,219,189 linhas)",1h 00min,32min 52s


✔ SVM binário: a CV de 5 dobras cabe (~2h 26min).
✔ Previsão do teste: ~32min 52s


## 5. Gravação e conferências
Grava as tabelas e o `calibracao_cache.json`, e confere o essencial: **o resultado do SVM não depende do cache** (mesmo número de vetores de suporte e mesmo ROC-AUC em todos os caches) e o treino em paralelo dá o mesmo modelo que o sequencial.

In [12]:
# ---------- 5.1 Gravação e conferências ----------
tab_cache.to_csv(PASTA / "cache_vs_tempo.csv", index=False); tab_par.to_csv(PASTA / "paralelismo_treino_com_cache.csv", index=False); tab_est2.drop(columns=["1_processo_s", "paralelo_s"]).to_csv(PASTA / "estimativas_atualizadas.csv", index=False)
hw = {"so": platform.platform(), "python": sys.version.split()[0], "nucleos_logicos": os.cpu_count()}
if psutil: hw.update({"nucleos_fisicos": psutil.cpu_count(logical=False), "ram_total_gib": psutil.virtual_memory().total / 2 ** 30})
salvar_json({"run_id": RUN_ID, "data": datetime.now().isoformat(timespec="seconds"), "hardware": hw, "tamanho_testado": n_c, "caches_mb": CACHES_MB, "cache_recomendado_mb": CACHE_REC, "tempo_com_cache_recomendado_s": T_REF, "n_sv": NSV_REF,
             "treinos_simultaneos_recomendados": K_REC, "ganho_treino": GANHO_TREINO, "expoente_tempo_suposto": EXPOENTE_TEMPO, "estimativas_horas": E2, "calibracao_usada": str(cals[-1]),
             "observacao": "complemento da calibração: testa a hipótese de que o cache de 200 MiB era o gargalo do treino com 81 mil linhas"}, PASTA / "calibracao_cache.json")
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
verifica("Pelo menos um tamanho de cache concluiu", len(ok_c) >= 1, f"({len(ok_c)} de {len(tab_cache)})")
verifica("O resultado do SVM não depende do cache: mesmo número de vetores de suporte em todos", ok_c.n_sv.nunique() <= 1 if len(ok_c) else False)
verifica("O resultado do SVM não depende do cache: mesmo ROC-AUC em todos", bool((ok_c.roc_auc_val.max() - ok_c.roc_auc_val.min()) < 1e-9) if len(ok_c) else False)
verifica("Treinos simultâneos dão o mesmo modelo (mesmo número de vetores de suporte)", bool(tab_par.n_sv_igual_em_todos.all()))
verifica("Estimativas finitas e positivas", bool(np.isfinite(tab_est2[["1_processo_s", "paralelo_s"]].to_numpy()).all() and (tab_est2[["1_processo_s", "paralelo_s"]].to_numpy() > 0).all()))
verifica("Arquivos gravados", all((PASTA / f).exists() for f in ("cache_vs_tempo.csv", "paralelismo_treino_com_cache.csv", "estimativas_atualizadas.csv", "calibracao_cache.json")))
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas. Resultados em {PASTA}")

✔ Pelo menos um tamanho de cache concluiu (2 de 4)
✔ O resultado do SVM não depende do cache: mesmo número de vetores de suporte em todos 
✔ O resultado do SVM não depende do cache: mesmo ROC-AUC em todos 
✔ Treinos simultâneos dão o mesmo modelo (mesmo número de vetores de suporte) 
✔ Estimativas finitas e positivas 
✔ Arquivos gravados 

✔ 6/6 checagens aprovadas. Resultados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_binario_calibracao\svm_cache_20261024_233500
